# Arabic Fake News Detection — Unified Research Pipeline

**Paper:** *Linguistic Markers of Deception in Arabic News Headlines: A Cross-Corpus Study of Stylistic and Numeric Features*  


## Pipeline Overview

| Stage | Dataset | Key Steps |
|-------|---------|-----------|
| **0** | — | Install dependencies, shared imports & utilities |
| **1a** | WELFake + LIAR | Translate EN→AR (Helsinki-NLP) |
| **1b** | WELFake + LIAR | Merge translated corpora |
| **1c** | WELFake + LIAR | Preprocess + extract features |
| **1d** | WELFake + LIAR | Train linear models (TF-IDF + LR / SVC) |
| **1e** | WELFake + LIAR | Fine-tune transformers (AraELECTRA, MARBERT) |
| **2a** | ANS | Preprocess official splits + extract features |
| **2b** | ANS | Train linear + transformer models |
| **3a** | VERA-ARAB | Preprocess + extract features |
| **3b** | VERA-ARAB | Train linear + transformer models |
| **4a** | AFND | Preprocess + extract features |
| **4b** | AFND | Train linear + transformer models |
| **5a** | Unified Corpus | Build & preprocess merged corpus |
| **5b** | Unified Corpus | Feature ranking (Mutual Information) |
| **5c** | Unified Corpus | TF-IDF linear ablation |
| **5d** | Unified Corpus | Transformer ablation (3 seeds) |
| **5e** | Unified Corpus | McNemar tests + summary tables + figures |




Install Dependencies

In [ ]:
# Run this cell FIRST (Google Colab)
!pip install -q "transformers>=4.45.0" "datasets>=2.19.0" evaluate \
               scikit-learn pandas scipy matplotlib pyarrow accelerate \
               sentencepiece sacrebleu tqdm joblib torch

Shared Imports & Device Setup

In [ ]:
import os, re, gc, json, math, random, warnings, time, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from scipy.sparse import hstack, vstack, csr_matrix
from scipy.stats import binomtest

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSequenceClassification, AutoConfig,
    PreTrainedModel, Trainer, TrainingArguments,
    TrainerCallback, pipeline as hf_pipeline
)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import __version__ as HF_VERSION

warnings.filterwarnings('ignore')

# Auto-detect eval_strategy key (renamed in transformers >= 4.45)
import inspect as _inspect
_ta_keys = set(_inspect.signature(TrainingArguments.__init__).parameters)
EVAL_KEY = 'eval_strategy' if 'eval_strategy' in _ta_keys else 'evaluation_strategy'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Python      : {sys.version.split()[0]}')
print(f'PyTorch     : {torch.__version__}')
print(f'Transformers: {HF_VERSION}')
print(f'Device      : {DEVICE}')
print(f'Eval key    : {EVAL_KEY!r}')

General Helper Functions

In [ ]:
def set_seed(seed: int = 42):
    """Fix all random seeds for reproducibility."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def log(msg: str):
    """Timestamped console logger."""
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

def save_json(path: str, obj):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def read_csv_robust(path: str, **kwargs) -> pd.DataFrame:
    """Try UTF-8, UTF-8-BOM, and Windows-1256 encodings in order."""
    for enc in ('utf-8', 'utf-8-sig', 'cp1256'):
        try:
            return pd.read_csv(path, encoding=enc, dtype=str, **kwargs)
        except Exception:
            pass
    raise FileNotFoundError(f'Could not read: {path}')

def makedirs(*paths):
    for p in paths:
        os.makedirs(p, exist_ok=True)

def plot_confusion_matrix(cm, title: str, save_path: str):
    """Save a labelled confusion matrix to disk."""
    fig, ax = plt.subplots(figsize=(4.8, 4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Real', 'Fake']); ax.set_yticklabels(['Real', 'Fake'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title, fontsize=9, fontweight='bold')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f'{v:,}', ha='center', va='center',
                color='white' if v > cm.max() * 0.6 else 'black', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.close()

print('Helper functions defined.')

Arabic Text Preprocessing

**Preprocessing philosophy
- Light normalisation — **preserve** stylistic cues (punctuation, numerals, quotation marks, stopwords, dialectal tokens)
- Replace volatile entities (URLs, emails, mentions, dates, money) with typed placeholders: `<URL>`, `<DATE>`, `<MONEY>`, etc.
- Unify Arabic letter variants; remove diacritics and tatweel


In [ ]:
# ── Compiled regex patterns ──────────────────────────────────────────────
AR_LETTERS    = r'\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF'
DIACRITICS    = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]')
TATWEEL       = re.compile(r'\u0640')
E2L_DIGITS    = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
URL_RE        = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE      = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')
MENTION_RE    = re.compile(r'(?<!\w)@\w+')
HASHTAG_RE    = re.compile(r'(?<!\w)#\w+')
DATE_RE       = re.compile(
    r'\b(?:\d{1,2}[-/\.]){{1,2}}\d{{2,4}}\b'
    r'|\b(يناير|فبراير|مارس|أبريل|مايو|يونيو|يوليو|أغسطس'
    r'|سبتمبر|أكتوبر|نوفمبر|ديسمبر|Jan|Feb|Mar|Apr|May|Jun'
    r'|Jul|Aug|Sep|Oct|Nov|Dec)\b', re.I
)
TIME_RE       = re.compile(r'\b([01]?\d|2[0-3]):[0-5]\d(:[0-5]\d)?\b')
PCT_RE        = re.compile(r'\d+\s?%')
MONEY_RE      = re.compile(
    r'(\$|€|£|ر\.?س\.?|ج\.?د\.?|د\.?ا\.?)\s?\d+(?:[\.,]\d+)?', re.UNICODE
)
QM_RE         = re.compile(r'[؟?]')
EXCL_RE       = re.compile(r'!')
EN_TOKEN_RE   = re.compile(r'^[A-Za-z]{2,}$')
ALLCAPS_RE    = re.compile(r'^[A-Z]{3,}$')
YEAR_RE       = re.compile(r'\b(19|20)\d{2}\b')
NON_LETTER_RE = re.compile(rf'[^A-Za-z{AR_LETTERS}\s]+')
MULTISPACE_RE = re.compile(r'\s+')

# ── Linguistic lexicons ───────────────────────────────────────────────────
NEGATIONS   = {'لا','ليس','لم','لن','بدون','غير'}
MODALS      = {'قد','ربما','يمكن','يُمكن','يمكنُ'}
HEDGES      = {'يقال','مزاعم','مزعوم','وفقاً','بحسب','تقرير','تقارير','مصادر','زعم'}
TRIGGERS    = {'عاجل','حصري','بالصور','بالفيديو','لن تصدق','صادم','فضيحة'}
FIRST_PERS  = {'انا','إني','نحن','لدينا','عندي','عندنا'}
HYPERBOLE   = {'الأعظم','الأقوى','الأكبر','غير مسبوق','هائل','مذهل','كارثي'}
DIALECT     = {'عايز','بتاع','شو','كمين','كتير','لهلأ','لسّا','هيك'}
EMOTION_POS = {'سعيد','مسرور','فرح','أمل','تفاؤل','نجاح'}
EMOTION_NEG = {'حزين','غاضب','خوف','قلق','كارثة','مصيبة','صادم'}
PAST_MARK   = {'كان','كانت','كنا','قال','صرّح','أعلن'}
PRES_MARK   = {'يقول','يعلن','يؤكد','تفيد','تحدث'}
FUT_MARK    = {'سوف','سي','سن','ست','س'}

def normalize_letters(s: str) -> str:
    """Unify Arabic letter variants (alef, taa marbuta, yaa, hamza)."""
    s = re.sub(r'[ٱآإأ]', 'ا', s)
    return s.replace('ة','ه').replace('ى','ي').replace('ؤ','و').replace('ئ','ي')

def preprocess_arabic(text) -> str:
    """
    Full Arabic-aware preprocessing:
    1) Replace entities with typed placeholders
    2) Unify Eastern Arabic digits to Latin
    3) Strip diacritics and tatweel
    4) Normalise letter variants
    5) Collapse excessive elongations (>=3 -> 2)
    6) Normalise whitespace
    """
    if pd.isna(text): return ''
    t = str(text)
    for pat, rep in [
        (URL_RE,'<URL>'), (EMAIL_RE,'<EMAIL>'), (MENTION_RE,'<USER>'),
        (HASHTAG_RE,'<HASHTAG>'), (PCT_RE,'<PCT>'), (MONEY_RE,'<MONEY>'),
        (DATE_RE,'<DATE>'), (TIME_RE,'<TIME>'),
    ]:
        t = pat.sub(rep, t)
    t = t.translate(E2L_DIGITS)
    t = DIACRITICS.sub('', t)
    t = TATWEEL.sub('', t)
    t = normalize_letters(t)
    t = re.sub(r'(.)\1{2,}', r'\1\1', t)
    return MULTISPACE_RE.sub(' ', t).strip()

def normalize_label(v):
    """Map raw label values to 'Fake' / 'Real'. Returns None for unknowns."""
    if pd.isna(v): return None
    s = str(v).strip().lower()
    TRUE_SET = {'true','real','valid','1','yes','y','t','real_news'}
    FAKE_SET = {'fake','false','misleading','hoax','0','no','n','f','fake_news'}
    if s in TRUE_SET: return 'Real'
    if s in FAKE_SET: return 'Fake'
    if 'fake' in s or 'false' in s: return 'Fake'
    if 'true' in s or 'real' in s: return 'Real'
    return None

print('Arabic preprocessing functions defined.')

Linguistic Feature Engineering


- **Length & density:** char/token count, avg/median word length, Yule's K, Guiraud's R, TTR
- **Numeric / temporal:** digit count, dates, years, times, money, percentages
- **Punctuation & style:** ?, !, ellipsis, multi-punct runs, quotation marks
- **Cross-script:** Latin %, Arabic %, ALL-CAPS tokens, English tokens
- **Discourse / lexicon:** negations, modals, hedges, sensational triggers, tense markers
- **Social / web artefacts:** URLs, hashtags, mentions


In [ ]:
def yule_k(tokens: list) -> float:
    """
    Yule's characteristic constant K — measures lexical repetition.
    Higher K = more repetition (lower diversity). (Eq. 1 in paper)
    K = 10^4 * (sum(i^2 * fi) - N) / N^2
    """
    N = len(tokens)
    if N == 0: return 0.0
    freq = Counter(tokens)
    freq_of_freq = Counter(freq.values())
    sum_i2fi = sum(i*i*fi for i, fi in freq_of_freq.items())
    return 1e4 * (sum_i2fi - N) / (N * N) if N > 1 else 0.0

def guiraud_r(tokens: list) -> float:
    """Guiraud's R = types / sqrt(tokens). Lexical richness measure."""
    N = len(tokens)
    return len(set(tokens)) / math.sqrt(N) if N > 0 else 0.0

def extract_features(text: str) -> dict:
    """
    Extract the full set of engineered linguistic and stylistic features
    from a single preprocessed Arabic headline.
    """
    words  = text.split()
    nw     = len(words)
    nc     = len(text)
    wlens  = [len(w) for w in words] or [0]
    uniq   = set(words)
    low    = text.lower()
    toks_low = low.split()

    # Length & density
    ttr_val     = len(uniq)/nw if nw else 0.0
    hapax_ratio = sum(1 for w in uniq if words.count(w)==1)/max(1,nw)
    guiraud     = guiraud_r(words)
    yule        = yule_k(words)

    # Character composition
    ar_chars  = len(re.findall(rf'[{AR_LETTERS}]', text))
    lat_chars = len(re.findall(r'[A-Za-z]', text))
    dig_chars = len(re.findall(r'\d', text))
    pct_chars = len(re.findall(r'[^\w\s<>]', text))

    # Numeric / temporal
    n_numbers   = len(re.findall(r'\d+', text))
    n_4dig_year = int(bool(YEAR_RE.search(text)))
    n_dates     = text.count('<DATE>')
    n_times     = text.count('<TIME>')
    n_money     = text.count('<MONEY>')
    n_pct       = text.count('<PCT>')

    # Punctuation & style
    has_q        = int(bool(QM_RE.search(text)))
    has_ex       = int(bool(EXCL_RE.search(text)))
    n_quotes     = sum(text.count(ch) for ch in ['"','«','»','\u201c','\u201d',"'"])
    has_quotes   = int(n_quotes > 0)
    has_ellipsis = int('…' in text or '...' in text)
    multi_punct  = int(len(re.findall(r'[\?!\.\,؛،:\-—]{3,}', text)) > 0)
    multi_exq    = int((len(QM_RE.findall(text))+len(EXCL_RE.findall(text))) >= 2)
    n_sym_runs   = len(re.findall(r'([*\/\\\|\_=\+\-]{3,})', text))
    punct_count  = pct_chars

    # Cross-script
    n_en_toks  = sum(1 for w in words if EN_TOKEN_RE.match(w))
    has_allcaps= int(any(ALLCAPS_RE.match(w) for w in words))

    # Discourse / lexicon
    n_neg    = sum(low.count(w) for w in NEGATIONS)
    n_modal  = sum(low.count(w) for w in MODALS)
    n_hedge  = sum(low.count(w) for w in HEDGES)
    has_trig = int(any(w in low for w in TRIGGERS))
    n_trig   = sum(low.count(w) for w in TRIGGERS)
    n_fp     = sum(toks_low.count(w) for w in FIRST_PERS)
    n_hyp    = sum(low.count(w) for w in HYPERBOLE)
    n_dial   = sum(low.count(w) for w in DIALECT)
    n_epos   = sum(t in EMOTION_POS for t in toks_low)
    n_eneg   = sum(t in EMOTION_NEG for t in toks_low)
    sentiment= (n_epos - n_eneg) / max(1, nw)
    has_past = int(any(w in toks_low for w in PAST_MARK))
    has_pres = int(any(w in toks_low for w in PRES_MARK))
    has_fut  = int(any(w in toks_low or bool(re.search(r'\bس\w+',low)) for w in FUT_MARK))
    rep_letters = int(bool(re.search(r'(.)\1{2,}', text)))
    rep_words   = int(bool(re.search(r'\b(\w+)\s+\1\b', text)))

    # Web / social
    n_url   = text.count('<URL>');   n_email = text.count('<EMAIL>')
    n_user  = text.count('<USER>');  n_hash  = text.count('<HASHTAG>')

    return {
        'len_chars':nc,'len_words':nw,'avg_word_len':float(np.mean(wlens)),
        'median_word_len':float(np.median(wlens)),
        'std_word_len':float(np.std(wlens)) if len(wlens)>1 else 0.0,
        'uniq_words':len(uniq),'ttr':ttr_val,'hapax_ratio':hapax_ratio,
        'guiraud_r':guiraud,'yule_k':yule,'space_count':text.count(' '),
        'pct_arabic':ar_chars/max(1,nc),'pct_latin':lat_chars/max(1,nc),
        'pct_digits':dig_chars/max(1,nc),'pct_punct':pct_chars/max(1,nc),
        'pct_spaces':text.count(' ')/max(1,nc),'punct_count':punct_count,
        'n_numbers':n_numbers,'has_4digit_year':n_4dig_year,
        'n_dates':n_dates,'n_times':n_times,'n_money':n_money,'n_pct':n_pct,
        'has_qmark':has_q,'has_exclam':has_ex,'multi_exq':multi_exq,
        'n_quotes':n_quotes,'has_quotes':has_quotes,
        'has_ellipsis':has_ellipsis,'multi_punct':multi_punct,'n_sym_runs':n_sym_runs,
        'n_en_tokens':n_en_toks,'has_allcaps':has_allcaps,
        'n_negations':n_neg,'n_modals':n_modal,'n_hedges':n_hedge,
        'has_trigger':has_trig,'n_triggers':n_trig,
        'n_first_person':n_fp,'n_hyperbole':n_hyp,'n_dialect':n_dial,
        'n_emotion_pos':n_epos,'n_emotion_neg':n_eneg,'sentiment':sentiment,
        'has_past':has_past,'has_present':has_pres,'has_future':has_fut,
        'rep_letters':rep_letters,'rep_words':rep_words,
        'n_url':n_url,'n_email':n_email,'n_user':n_user,'n_hash':n_hash,
    }

print('Feature extraction functions defined.')

Shared Modelling Utilities (Metrics, MI Ranking, TF-IDF)

In [ ]:
# ── Metrics ──────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred) -> dict:
    yt, yp = np.array(y_true), np.array(y_pred)
    return {
        'Accuracy':          accuracy_score(yt, yp),
        'Precision (macro)': precision_score(yt, yp, average='macro', zero_division=0),
        'Recall (macro)':    recall_score(yt, yp,    average='macro', zero_division=0),
        'F1 (macro)':        f1_score(yt, yp,        average='macro', zero_division=0),
    }

def mcnemar_exact(y_true, pred_a, pred_b) -> dict:
    """McNemar's exact test (two-sided binomial). Returns p-value and significance flag."""
    ya, yb, yt = np.array(pred_a), np.array(pred_b), np.array(y_true)
    ok_a = (ya==yt); ok_b = (yb==yt)
    b = int(np.sum(ok_a & ~ok_b)); c = int(np.sum(~ok_a & ok_b))
    n = b + c
    p = 1.0 if n==0 else binomtest(min(b,c), n, 0.5, alternative='two-sided').pvalue
    return {'b':b,'c':c,'n_discordant':n,'p_value':float(p),'significant':bool(p<0.05)}

def bootstrap_ci(y_true, y_pred, metric='f1_macro', n=2000, alpha=0.05, seed=0):
    """Return (mean, lower_CI, upper_CI) via bootstrap resampling."""
    rng = np.random.default_rng(seed)
    yt, yp, N = np.array(y_true), np.array(y_pred), len(y_true)
    scores = []
    for _ in range(n):
        ix = rng.integers(0, N, N)
        if metric=='f1_macro':
            scores.append(f1_score(yt[ix], yp[ix], average='macro', zero_division=0))
        else:
            scores.append(accuracy_score(yt[ix], yp[ix]))
    lo, hi = np.percentile(scores, [100*alpha/2, 100*(1-alpha/2)])
    return float(np.mean(scores)), float(lo), float(hi)

def majority_vote(pred_list):
    return (np.stack(pred_list).mean(0) >= 0.5).astype(int)

def get_warmup_steps(n_train, batch_size, grad_accum, epochs, warmup_ratio=0.06):
    steps = math.ceil(n_train / (batch_size * grad_accum)) * epochs
    return max(1, int(steps * warmup_ratio))

# ── Feature ranking ───────────────────────────────────────────────────────
def rank_features_by_mi(feat_df, labels, exclude_cols, top_k=10):
    """Rank features by mutual information on the train split ONLY."""
    num_cols = [c for c in feat_df.columns
                if c not in exclude_cols and pd.api.types.is_numeric_dtype(feat_df[c])]
    X  = feat_df[num_cols].fillna(0).to_numpy(dtype=np.float32)
    mi = mutual_info_classif(X, labels, random_state=42)
    ranked = sorted(zip(num_cols, mi), key=lambda x: x[1], reverse=True)
    top_names = [n for n,_ in ranked[:top_k]]
    mi_vals   = [v for _,v in ranked[:top_k]]
    return top_names, mi_vals, ranked

# ── TF-IDF builder ────────────────────────────────────────────────────────
def build_tfidf(word_ngrams=(1,2), char_ngrams=(3,6)):
    """Word (1-2) + char_wb (3-6) TF-IDF union — matches paper config."""
    return ColumnTransformer(
        transformers=[
            ('w', TfidfVectorizer(analyzer='word', ngram_range=word_ngrams,
                                  min_df=2, max_df=0.95, sublinear_tf=True, lowercase=False),
             'text_clean'),
            ('c', TfidfVectorizer(analyzer='char_wb', ngram_range=char_ngrams,
                                  min_df=2, max_df=1.0, sublinear_tf=True, lowercase=False),
             'text_clean'),
        ],
        remainder='drop', sparse_threshold=1.0
    )

# ── Linear model runner ───────────────────────────────────────────────────
def run_linear_model(name, estimator, X_train, y_train, X_dev, y_dev,
                     X_test, y_test, C_grid, report_dir, fig_dir):
    """Train LR or SVC with C tuned on dev; refit on train+dev; evaluate on test."""
    best_C, best_f1 = None, -1.0
    for C in C_grid:
        est = clone(estimator); est.set_params(C=C)
        est.fit(X_train, y_train)
        f1 = f1_score(y_dev, est.predict(X_dev), average='macro', zero_division=0)
        if f1 > best_f1: best_f1, best_C = f1, C
    Xfull = vstack([X_train, X_dev]) if hasattr(X_train,'toarray') else np.vstack([X_train, X_dev])
    yfull = np.concatenate([y_train, y_dev])
    best = clone(estimator); best.set_params(C=best_C); best.fit(Xfull, yfull)
    y_pred = best.predict(X_test)
    m   = compute_metrics(y_test, y_pred)
    cm  = confusion_matrix(y_test, y_pred)
    rep = classification_report(y_test, y_pred, digits=4)
    makedirs(report_dir, fig_dir)
    with open(os.path.join(report_dir, f'{name}.txt'), 'w') as f:
        f.write(f'{name} (C={best_C})\n\n{rep}\nConfusion Matrix:\n{cm}\n')
    plot_confusion_matrix(cm, name, os.path.join(fig_dir, f'{name}_cm.png'))
    log(f'[{name}] Best C={best_C} | Test macro-F1={m["F1 (macro)"]:.4f}')
    return {**m, 'model':name, 'best_C':best_C, 'y_pred':y_pred}

print('Shared modelling utilities defined.')

Transformer Components

- **`HFTextDataset`** — wraps tokenised text (+ optional Top-K features) for the Trainer
- **`FusionClassifier`** — backbone + small MLP on engineered features → late fusion
- **`HistoryCallback`** — logs training loss for loss-curve plotting


In [ ]:
class HFTextDataset(Dataset):
    """
    HuggingFace Trainer-compatible dataset.
    Optional 'feats' array enables late-fusion experiments.
    """
    def __init__(self, enc, labels, feats=None):
        self.enc=enc; self.labels=labels; self.feats=feats
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.enc.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        if self.feats is not None:
            item['extra_feats'] = torch.tensor(self.feats[idx], dtype=torch.float32)
        return item

class FusionClassifier(nn.Module):
    """
    Late-fusion model: transformer backbone + MLP on Top-K engineered features.
    Concatenates pooled CLS with MLP output before the final linear classifier.
    Returns SequenceClassifierOutput so standard Trainer handles it.
    """
    def __init__(self, hub:str, extra_dim:int, num_labels:int=2, class_weights=None):
        super().__init__()
        self.backbone  = AutoModel.from_pretrained(hub)
        hidden         = self.backbone.config.hidden_size
        self.extra_dim = extra_dim
        mlp_out        = max(8, extra_dim)
        self.feat_mlp  = (nn.Sequential(nn.Linear(extra_dim, mlp_out), nn.ReLU())
                          if extra_dim>0 else None)
        self.classifier= nn.Linear(hidden+(mlp_out if extra_dim>0 else 0), num_labels)
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights.clone().detach())
        else:
            self.class_weights = None

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                extra_feats=None, labels=None, **kwargs):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
                            token_type_ids=token_type_ids, return_dict=True)
        pooled = (out.pooler_output if getattr(out,'pooler_output',None) is not None
                  else out.last_hidden_state[:,0])
        if self.extra_dim>0 and extra_feats is not None:
            pooled = torch.cat([pooled, self.feat_mlp(extra_feats)], dim=-1)
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            cw = (self.class_weights.to(logits.device)
                  if self.class_weights is not None else None)
            loss = nn.CrossEntropyLoss(weight=cw)(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)

class HistoryCallback(TrainerCallback):
    def __init__(self): self.history=[]
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs: self.history.append({**logs,'step':state.global_step})

def run_transformer(run_name, hub, tr_texts, dv_texts, te_texts,
                    y_tr, y_dv, y_te, topk_pack,
                    report_dir, fig_dir, ckpt_dir,
                    max_len=192, epochs=4, lr=2e-5, bsz=32,
                    grad_accum=1, seed=42):
    """
    Train (or resume from checkpoint) one transformer run and evaluate on test.
    topk_pack: dict('train','dev','test') of float32 arrays, or None for text-only.
    """
    set_seed(seed)
    makedirs(ckpt_dir, report_dir, fig_dir)
    pred_path = os.path.join(report_dir, f'{run_name}_preds.npy')
    if os.path.exists(pred_path):
        log(f'  [CACHED] {run_name}')
        y_pred = np.load(pred_path).tolist()
        m = compute_metrics(y_te, y_pred)
        log(f'    Acc={m["Accuracy"]:.4f}  F1={m["F1 (macro)"]:.4f}')
        return {**m,'model':run_name,'y_pred':np.array(y_pred)}
    log(f'  Training {run_name} …')
    tok = AutoTokenizer.from_pretrained(hub)
    def tokenise(txts):
        return tok(list(txts), padding=True, truncation=True,
                   max_length=max_len, return_attention_mask=True)
    tr_enc=tokenise(tr_texts); dv_enc=tokenise(dv_texts); te_enc=tokenise(te_texts)
    cc = pd.Series(y_tr).value_counts().sort_index()
    cw = torch.tensor((cc.sum()/(2.0*cc)).to_numpy(dtype=np.float32))
    warmup = get_warmup_steps(len(y_tr), bsz, grad_accum, epochs)
    if topk_pack is None:
        tr_ds=HFTextDataset(tr_enc,y_tr); dv_ds=HFTextDataset(dv_enc,y_dv)
        te_ds=HFTextDataset(te_enc,y_te)
        cfg_m=AutoConfig.from_pretrained(hub,num_labels=2,
                                          id2label={0:'real',1:'fake'},
                                          label2id={'real':0,'fake':1})
        model=AutoModelForSequenceClassification.from_pretrained(hub,config=cfg_m)
    else:
        tr_ds=HFTextDataset(tr_enc,y_tr,feats=topk_pack['train'].astype(np.float32))
        dv_ds=HFTextDataset(dv_enc,y_dv,feats=topk_pack['dev'].astype(np.float32))
        te_ds=HFTextDataset(te_enc,y_te,feats=topk_pack['test'].astype(np.float32))
        model=FusionClassifier(hub,topk_pack['train'].shape[1],num_labels=2,class_weights=cw)
    hist_cb = HistoryCallback()
    ta_kwargs = {
        'output_dir':ckpt_dir,'num_train_epochs':epochs,
        'per_device_train_batch_size':bsz,'per_device_eval_batch_size':bsz,
        'gradient_accumulation_steps':grad_accum,'learning_rate':lr,
        'weight_decay':0.01,'warmup_steps':warmup,
        'save_strategy':'epoch',EVAL_KEY:'epoch',
        'load_best_model_at_end':True,'metric_for_best_model':'eval_loss',
        'greater_is_better':False,'save_total_limit':2,'logging_steps':100,
        'report_to':[],'fp16':torch.cuda.is_available() and (topk_pack is None),
    }
    trainer=Trainer(model=model,args=TrainingArguments(**ta_kwargs),
                    train_dataset=tr_ds,eval_dataset=dv_ds,callbacks=[hist_cb])
    trainer.train()
    pred  = trainer.predict(te_ds)
    y_pred= pred.predictions.argmax(axis=-1).tolist()
    np.save(pred_path, np.array(y_pred))
    m  = compute_metrics(y_te, y_pred)
    cm = confusion_matrix(y_te, y_pred)
    rep= classification_report(y_te, y_pred, digits=4, target_names=['Real','Fake'])
    with open(os.path.join(report_dir,f'{run_name}.txt'),'w') as f:
        f.write(f'{run_name}\n\n{rep}\nConfusion Matrix:\n{cm}\n')
    plot_confusion_matrix(cm,run_name,os.path.join(fig_dir,f'{run_name}_cm.png'))
    save_json(os.path.join(report_dir,f'{run_name}_history.json'),hist_cb.history)
    log(f'    Acc={m["Accuracy"]:.4f}  F1={m["F1 (macro)"]:.4f}')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {**m,'model':run_name,'y_pred':np.array(y_pred),'CM':cm}

print('Transformer components defined.')

---
WELFake + LIAR (English → Arabic Translation)



LIAR and WELFake are English datasets. We translate them to Arabic using  
`Helsinki-NLP/opus-mt-en-ar` (MarianMT) so they can join the Arabic corpora.




Set WELFake + LIAR Paths

In [ ]:
# ── Edit these paths to match your uploaded files ────────────────────────
WL_BASE_DIR  = '/content/extracted/LINGUESTIC DATASET/WELFAKE'
LIAR_IN_PATH = '/content/extracted/LINGUESTIC DATASET/LIAR/merged_label_statement_binary_ar.csv'
LIAR_OUT_PATH= '/content/extracted/LINGUESTIC DATASET/LIAR/merged_label_statement_binary_ar_HELSINKI_MT.csv'
WL_MERGED    = '/content/Merged_WELFAKE_AND_LIAR_HELSINKI_MT.csv'
WL_CLEAN     = '/content/Clean_WELFAKE_AND_LIAR_HELSINKI_MT.csv'
WL_FEAT      = '/content/Features_WELFAKE_AND_LIAR_HELSINKI_MT.csv'
WL_REPORTS   = '/content/reports/welfake_liar'
WL_FIGS      = '/content/figs/welfake_liar'
makedirs(WL_REPORTS, WL_FIGS)
print('Paths set.')

Translate WELFake (Parts 01–72) to Arabic

In [ ]:
# Translates each welfake_part_XX_arabic.csv → welfake_part_XX_arabic_HELSINKI_MT.csv
# Skips parts that already have an output file (resumable).

WELFAKE_BATCH_SIZE = 128   # reduce to 64 if OOM
WELFAKE_CHUNK_SIZE = 5000
WELFAKE_START_PART = 1
WELFAKE_END_PART   = 72

device_id = 0 if torch.cuda.is_available() else -1
translator = hf_pipeline(
    task='translation_en_to_ar',
    model='Helsinki-NLP/opus-mt-en-ar',
    device=device_id
)

def translate_list(texts):
    out = []
    for i in range(0, len(texts), WELFAKE_BATCH_SIZE):
        out.extend([r['translation_text']
                    for r in translator(texts[i:i+WELFAKE_BATCH_SIZE],
                                        clean_up_tokenization_spaces=True)])
    return out

for part in range(WELFAKE_START_PART, WELFAKE_END_PART + 1):
    in_path  = os.path.join(WL_BASE_DIR, f'welfake_part_{part:02d}_arabic.csv')
    out_path = os.path.join(WL_BASE_DIR, f'welfake_part_{part:02d}_arabic_HELSINKI_MT.csv')
    if not os.path.exists(in_path):
        print(f'  [SKIP] Not found: {in_path}'); continue
    if os.path.exists(out_path):
        print(f'  [SKIP] Already translated: part {part:02d}'); continue
    print(f'  Translating part {part:02d} …')
    first = True
    for enc in ('utf-8','utf-8-sig','cp1256'):
        try:
            df_iter = pd.read_csv(in_path, chunksize=WELFAKE_CHUNK_SIZE, encoding=enc, dtype=str)
            break
        except Exception: continue
    for chunk in df_iter:
        chunk['HELSINKI_MT'] = translate_list(chunk['title'].fillna('').astype(str).tolist())
        chunk.to_csv(out_path, mode='w' if first else 'a', header=first,
                     index=False, encoding='utf-8-sig')
        first = False
    print(f'  Saved: {out_path}')

print('WELFake translation complete.')

Translate LIAR Statements to Arabic

In [ ]:
# Translates 'statement' column in LIAR CSV → adds HELSINKI_MT column

LIAR_BATCH_SIZE = 128
LIAR_CHUNK_SIZE = 5000

if os.path.exists(LIAR_OUT_PATH):
    print('[SKIP] LIAR already translated.')
else:
    first = True
    for enc in ('utf-8','utf-8-sig','cp1256'):
        try:
            df_iter = pd.read_csv(LIAR_IN_PATH, chunksize=LIAR_CHUNK_SIZE, encoding=enc, dtype=str)
            break
        except Exception: continue
    for chunk in df_iter:
        if 'statement' not in chunk.columns:
            raise KeyError("'statement' column not found in LIAR file")
        texts = chunk['statement'].fillna('').astype(str).tolist()
        translated = translate_list(texts)
        chunk['HELSINKI_MT'] = translated
        chunk.to_csv(LIAR_OUT_PATH, mode='w' if first else 'a', header=first,
                     index=False, encoding='utf-8-sig')
        first = False
    print(f'LIAR translation saved to: {LIAR_OUT_PATH}')

Merge WELFake + LIAR into One CSV

In [ ]:
# Collects columns [HELSINKI_MT, label] from all translated parts
# and combines them into a single binary-labelled file.

if os.path.exists(WL_MERGED):
    print(f'[SKIP] Merged CSV already exists: {WL_MERGED}')
else:
    frames = []
    for i in range(1, 73):
        p = os.path.join(WL_BASE_DIR, f'welfake_part_{i:02d}_arabic_HELSINKI_MT.csv')
        if not os.path.exists(p): continue
        df = read_csv_robust(p)
        df.columns = [c.strip() for c in df.columns]
        if 'HELSINKI_MT' in df.columns and 'label' in df.columns:
            frames.append(df[['HELSINKI_MT','label']].copy())
    if os.path.exists(LIAR_OUT_PATH):
        df_l = read_csv_robust(LIAR_OUT_PATH)
        df_l.columns = [c.strip() for c in df_l.columns]
        if 'HELSINKI_MT' in df_l.columns and 'label' in df_l.columns:
            frames.append(df_l[['HELSINKI_MT','label']].copy())
    merged = pd.concat(frames, ignore_index=True)
    merged['HELSINKI_MT'] = merged['HELSINKI_MT'].fillna('').astype(str)
    merged.to_csv(WL_MERGED, index=False, encoding='utf-8-sig')
    print(f'Merged {len(merged):,} rows → {WL_MERGED}')
    print(merged['label'].value_counts().to_dict())

Preprocess WELFake + LIAR & Extract Features

In [ ]:
# Applies Arabic preprocessing and extracts all linguistic features.
# Saves two aligned CSVs: clean text and per-row features.

def wl_label(x):
    s = str(x).strip().lower()
    if s in {'1','fake','f'}: return 'Fake'
    if s in {'0','real','r'}: return 'Real'
    return None

if os.path.exists(WL_CLEAN) and os.path.exists(WL_FEAT):
    log('[SKIP] WL clean/features CSVs already exist — loading …')
    df_wl_clean = pd.read_csv(WL_CLEAN, encoding='utf-8-sig')
    df_wl_feats = pd.read_csv(WL_FEAT,  encoding='utf-8-sig')
else:
    log('Preprocessing WELFake + LIAR …')
    df = read_csv_robust(WL_MERGED)
    df.columns = [c.strip() for c in df.columns]
    df['HELSINKI_MT'] = df['HELSINKI_MT'].fillna('').astype(str)
    df['label_str'] = df['label'].apply(wl_label)
    df = df.dropna(subset=['label_str']).reset_index(drop=True)

    clean_texts, feat_rows = [], []
    for raw in df['HELSINKI_MT']:
        clean = preprocess_arabic(raw)
        clean_texts.append(clean)
        feat_rows.append(extract_features(clean))

    df_wl_clean = pd.DataFrame({'text_clean': clean_texts, 'label': df['label_str']})
    df_wl_feats = pd.DataFrame(feat_rows)
    df_wl_feats['label'] = df['label_str'].values

    # Drop empty / short / duplicate rows
    mask = (df_wl_clean['text_clean'].str.strip() != '') & \
           (df_wl_clean['text_clean'].str.split().str.len() > 2)
    df_wl_clean = df_wl_clean.loc[mask].drop_duplicates(subset=['text_clean']).reset_index(drop=True)
    df_wl_feats = df_wl_feats.loc[mask].reset_index(drop=True).iloc[:len(df_wl_clean)]

    df_wl_clean.to_csv(WL_CLEAN, index=False, encoding='utf-8-sig')
    df_wl_feats.to_csv(WL_FEAT,  index=False, encoding='utf-8-sig')

log(f'WL corpus: {len(df_wl_clean):,} rows | {df_wl_clean["label"].value_counts().to_dict()}')

WELFake + LIAR: Train TF-IDF Linear Models

In [ ]:
# Stratified 80/10/10 split → feature ranking → TF-IDF + LR / SVC + Hybrid

set_seed(42)
df_wl_clean['y'] = df_wl_clean['label'].map({'Fake':1,'Real':0})
y_all_wl = df_wl_clean['y'].to_numpy()

# 80/10/10 split
idx_wl = np.arange(len(df_wl_clean))
tr_wl, tmp_wl = train_test_split(idx_wl, test_size=0.20, stratify=y_all_wl, random_state=42)
dv_wl, te_wl  = train_test_split(tmp_wl, test_size=0.50, stratify=y_all_wl[tmp_wl], random_state=42)

feat_wl = df_wl_feats.copy()
for c in feat_wl.columns:
    feat_wl[c] = pd.to_numeric(feat_wl[c], errors='coerce').fillna(0)

top_names_wl, mi_vals_wl, _ = rank_features_by_mi(
    feat_wl.iloc[tr_wl].reset_index(drop=True), y_all_wl[tr_wl], {'label'}, top_k=10)
log(f'Top-10 WL features: {top_names_wl}')

scaler_wl = StandardScaler().fit(feat_wl.iloc[tr_wl][top_names_wl].to_numpy(np.float32))
Xk_wl = {s: scaler_wl.transform(feat_wl.iloc[ix][top_names_wl].to_numpy(np.float32))
         for s, ix in [('train',tr_wl),('dev',dv_wl),('test',te_wl)]}

tr_txt_wl = df_wl_clean.iloc[tr_wl][['text_clean']].reset_index(drop=True)
dv_txt_wl = df_wl_clean.iloc[dv_wl][['text_clean']].reset_index(drop=True)
te_txt_wl = df_wl_clean.iloc[te_wl][['text_clean']].reset_index(drop=True)
y_tr_wl = y_all_wl[tr_wl]; y_dv_wl = y_all_wl[dv_wl]; y_te_wl = y_all_wl[te_wl]

feat_wl_obj = build_tfidf()
Xtr_wl_t = feat_wl_obj.fit_transform(tr_txt_wl)
Xdv_wl_t = feat_wl_obj.transform(dv_txt_wl)
Xte_wl_t = feat_wl_obj.transform(te_txt_wl)
Xtr_wl_h = hstack([Xtr_wl_t, csr_matrix(Xk_wl['train'])])
Xdv_wl_h = hstack([Xdv_wl_t, csr_matrix(Xk_wl['dev'])])
Xte_wl_h = hstack([Xte_wl_t, csr_matrix(Xk_wl['test'])])

C_GRID_WL = [0.0625, 0.125, 0.25, 0.5, 1, 2, 4]
lr_wl  = LogisticRegression(penalty='l2', solver='liblinear', class_weight='balanced', max_iter=2000)
svm_wl = LinearSVC(class_weight='balanced', max_iter=5000)

wl_linear_results = []
for name, est, Xtr, Xdv, Xte in [
    ('WL_TFIDF_LR',   lr_wl,  Xtr_wl_t, Xdv_wl_t, Xte_wl_t),
    ('WL_TFIDF_SVC',  svm_wl, Xtr_wl_t, Xdv_wl_t, Xte_wl_t),
    ('WL_Hybrid_LR',  lr_wl,  Xtr_wl_h, Xdv_wl_h, Xte_wl_h),
    ('WL_Hybrid_SVC', svm_wl, Xtr_wl_h, Xdv_wl_h, Xte_wl_h),
]:
    r = run_linear_model(name, est, Xtr, y_tr_wl, Xdv, y_dv_wl, Xte, y_te_wl,
                         C_GRID_WL, WL_REPORTS, WL_FIGS)
    wl_linear_results.append(r)

print('WL linear models done.')
pd.DataFrame([{k:v for k,v in r.items() if k not in ('y_pred','CM')} for r in wl_linear_results])

WELFake + LIAR: Fine-tune Transformers

In [ ]:
# Fine-tunes AraELECTRA and MARBERT, text-only and with Top-10 fusion.
# Cached: if prediction file exists, evaluation is reloaded without re-training.

WL_TRANSFORMER_MODELS = [
    ('WL_AraELECTRA', 'aubmindlab/araelectra-base-discriminator'),
    ('WL_MARBERT',    'UBC-NLP/MARBERT'),
]
wl_topk_pack = {'train': Xk_wl['train'], 'dev': Xk_wl['dev'], 'test': Xk_wl['test']}

tr_texts_wl = df_wl_clean.iloc[tr_wl]['text_clean'].tolist()
dv_texts_wl = df_wl_clean.iloc[dv_wl]['text_clean'].tolist()
te_texts_wl = df_wl_clean.iloc[te_wl]['text_clean'].tolist()

for base_name, hub in WL_TRANSFORMER_MODELS:
    for tag, pack in [('text', None), ('top10', wl_topk_pack)]:
        run_transformer(
            run_name=f'{base_name}_{tag}', hub=hub,
            tr_texts=tr_texts_wl, dv_texts=dv_texts_wl, te_texts=te_texts_wl,
            y_tr=y_tr_wl, y_dv=y_dv_wl, y_te=y_te_wl,
            topk_pack=pack,
            report_dir=WL_REPORTS, fig_dir=WL_FIGS,
            ckpt_dir=os.path.join(WL_REPORTS, 'checkpoints', f'{base_name}_{tag}'),
            max_len=192, epochs=4, lr=2e-5, bsz=32
        )

print('WL transformer fine-tuning complete.')

---
ANS Dataset


Uses the **official ANS train/dev/test splits** — no re-splitting.  
Columns: `claim_s` (text), `fake_flag` (0=real, 1=fake).  
Class imbalance ~1:2.08 (fake:real) — handled via `class_weight='balanced'`.


ANS: Set Paths & Preprocess All Splits

In [ ]:
# ── Edit paths ───────────────────────────────────────────────────────────
ANS_TRAIN = '/content/train.csv'
ANS_DEV   = '/content/dev.csv'
ANS_TEST  = '/content/test.csv'
ANS_REPORTS = '/content/reports/ans'
ANS_FIGS    = '/content/figs/ans'
makedirs(ANS_REPORTS, ANS_FIGS)

def ans_process_split(split_name, path):
    log(f'  Processing ANS {split_name} …')
    df = read_csv_robust(path)
    df.columns = [c.strip() for c in df.columns]
    df = df[['claim_s','fake_flag']].copy()
    def map_lbl(x):
        s = str(x).strip()
        return 'Real' if s=='0' else ('Fake' if s=='1' else None)
    df['label'] = df['fake_flag'].apply(map_lbl)
    df = df.dropna(subset=['label']).reset_index(drop=True)
    clean_texts, feat_rows = [], []
    for raw in df['claim_s']:
        clean = preprocess_arabic(raw)
        clean_texts.append(clean)
        feat_rows.append(extract_features(clean))
    df_clean = pd.DataFrame({'text_clean':clean_texts,'label':df['label']})
    df_feats = pd.DataFrame(feat_rows)
    df_feats['label'] = df['label'].values
    mask = (df_clean['text_clean'].str.strip()!='') & \
           (df_clean['text_clean'].str.split().str.len()>2)
    df_clean = df_clean.loc[mask].drop_duplicates(subset=['text_clean']).reset_index(drop=True)
    df_feats = df_feats.loc[mask].reset_index(drop=True).iloc[:len(df_clean)]
    df_clean.to_csv(os.path.join(ANS_REPORTS,f'Clean_ANS_{split_name}.csv'),
                    index=False, encoding='utf-8-sig')
    df_feats.to_csv(os.path.join(ANS_REPORTS,f'Features_ANS_{split_name}.csv'),
                    index=False, encoding='utf-8-sig')
    log(f'    {split_name}: {len(df_clean)} rows | {df_clean["label"].value_counts().to_dict()}')
    return df_clean, df_feats

df_ans_tr, ft_ans_tr = ans_process_split('train', ANS_TRAIN)
df_ans_dv, ft_ans_dv = ans_process_split('dev',   ANS_DEV)
df_ans_te, ft_ans_te = ans_process_split('test',  ANS_TEST)
print('ANS preprocessing complete.')

ANS: Feature Ranking (Mutual Information on Train)

In [ ]:
y_ans_tr = df_ans_tr['label'].map({'Fake':1,'Real':0}).to_numpy()
y_ans_dv = df_ans_dv['label'].map({'Fake':1,'Real':0}).to_numpy()
y_ans_te = df_ans_te['label'].map({'Fake':1,'Real':0}).to_numpy()

for df in (ft_ans_tr, ft_ans_dv, ft_ans_te):
    for c in df.columns:
        if c != 'label': df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

top_names_ans, mi_vals_ans, ranked_ans = rank_features_by_mi(
    ft_ans_tr.drop(columns=['label']), y_ans_tr, set(), top_k=10)
log(f'Top-10 ANS features: {top_names_ans}')

# Show per-class feature means (Table 5 in paper)
feat_display = ft_ans_tr.drop(columns=['label']).copy()
feat_display['label'] = df_ans_tr['label'].values
means_by_class = feat_display.groupby('label')[top_names_ans].mean().T
means_by_class.columns.name = None
means_by_class['|diff|'] = abs(means_by_class.get('Fake',0) - means_by_class.get('Real',0))
print(means_by_class.sort_values('|diff|', ascending=False).round(4).to_string())

 ANS: Train TF-IDF Linear Models

In [ ]:
scaler_ans = StandardScaler().fit(ft_ans_tr[top_names_ans].to_numpy(np.float32))
Xk_ans = {s: scaler_ans.transform(ft[top_names_ans].to_numpy(np.float32))
          for s, ft in [('train',ft_ans_tr),('dev',ft_ans_dv),('test',ft_ans_te)]}

tr_ans_txt = df_ans_tr[['text_clean']]; dv_ans_txt = df_ans_dv[['text_clean']]
te_ans_txt = df_ans_te[['text_clean']]

feat_ans = build_tfidf()
Xtr_ans_t = feat_ans.fit_transform(tr_ans_txt)
Xdv_ans_t = feat_ans.transform(dv_ans_txt)
Xte_ans_t = feat_ans.transform(te_ans_txt)
Xtr_ans_h = hstack([Xtr_ans_t, csr_matrix(Xk_ans['train'])])
Xdv_ans_h = hstack([Xdv_ans_t, csr_matrix(Xk_ans['dev'])])
Xte_ans_h = hstack([Xte_ans_t, csr_matrix(Xk_ans['test'])])

C_GRID_ANS = [0.0625, 0.125, 0.5, 1, 2, 4]
lr_ans  = LogisticRegression(penalty='l2', solver='liblinear', class_weight='balanced', max_iter=3000)
svm_ans = LinearSVC(class_weight='balanced', max_iter=5000)

ans_linear_results = []
for name, est, Xtr, Xdv, Xte in [
    ('ANS_TFIDF_LR',   lr_ans,  Xtr_ans_t, Xdv_ans_t, Xte_ans_t),
    ('ANS_TFIDF_SVC',  svm_ans, Xtr_ans_t, Xdv_ans_t, Xte_ans_t),
    ('ANS_Hybrid_LR',  lr_ans,  Xtr_ans_h, Xdv_ans_h, Xte_ans_h),
    ('ANS_Hybrid_SVC', svm_ans, Xtr_ans_h, Xdv_ans_h, Xte_ans_h),
]:
    r = run_linear_model(name, est, Xtr, y_ans_tr, Xdv, y_ans_dv, Xte, y_ans_te,
                         C_GRID_ANS, ANS_REPORTS, ANS_FIGS)
    ans_linear_results.append(r)
print('ANS linear models done.')

 ANS: Fine-tune Transformers

In [ ]:
ANS_TRANSFORMER_MODELS = [
    ('ANS_AraELECTRA', 'aubmindlab/araelectra-base-discriminator'),
    ('ANS_MARBERT',    'UBC-NLP/MARBERT'),
]
ans_topk_pack = {'train':Xk_ans['train'],'dev':Xk_ans['dev'],'test':Xk_ans['test']}

for base_name, hub in ANS_TRANSFORMER_MODELS:
    for tag, pack in [('text', None), ('top10', ans_topk_pack)]:
        run_transformer(
            run_name=f'{base_name}_{tag}', hub=hub,
            tr_texts=df_ans_tr['text_clean'].tolist(),
            dv_texts=df_ans_dv['text_clean'].tolist(),
            te_texts=df_ans_te['text_clean'].tolist(),
            y_tr=y_ans_tr, y_dv=y_ans_dv, y_te=y_ans_te,
            topk_pack=pack,
            report_dir=ANS_REPORTS, fig_dir=ANS_FIGS,
            ckpt_dir=os.path.join(ANS_REPORTS,'checkpoints',f'{base_name}_{tag}'),
            max_len=192, epochs=4, lr=2e-5, bsz=32
        )
print('ANS transformer fine-tuning complete.')

---
VERA-ARAB Dataset

**Section 3.3 of paper**  
Multi-dialectal Arabic tweets


 VERA-ARAB: Preprocess + Extract Features

In [ ]:
# ── Edit path ────────────────────────────────────────────────────────────
VA_CSV     = '/content/VERAARAB.csv'
VA_REPORTS = '/content/reports/vera_arab'
VA_FIGS    = '/content/figs/vera_arab'
makedirs(VA_REPORTS, VA_FIGS)
set_seed(42)

df_va = read_csv_robust(VA_CSV)
df_va.columns = [c.strip() for c in df_va.columns]

# Detect text and label columns
va_text_col  = next((c for c in df_va.columns if any(k in c.lower() for k in ['text','tweet'])), df_va.columns[0])
va_label_col = next((c for c in df_va.columns if any(k in c.lower() for k in ['label','class'])), df_va.columns[1])

df_va['text_clean'] = df_va[va_text_col].fillna('').apply(preprocess_arabic)
df_va['label_str']  = df_va[va_label_col].apply(normalize_label)
df_va = df_va.dropna(subset=['label_str'])
df_va = df_va[df_va['text_clean'].str.split().str.len()>2]
df_va = df_va.drop_duplicates(subset=['text_clean']).reset_index(drop=True)
df_va['y'] = df_va['label_str'].map({'Fake':1,'Real':0})

ft_va = pd.DataFrame([extract_features(t) for t in df_va['text_clean']])

log(f'VERA-ARAB: {len(df_va):,} rows | {df_va["label_str"].value_counts().to_dict()}')

VERA-ARAB: Train Linear + Transformer Models

In [ ]:
y_va = df_va['y'].to_numpy()
idx_va = np.arange(len(df_va))
tr_va, tmp_va = train_test_split(idx_va, test_size=0.20, stratify=y_va, random_state=42)
dv_va, te_va  = train_test_split(tmp_va, test_size=0.50, stratify=y_va[tmp_va], random_state=42)

top_names_va, _, _ = rank_features_by_mi(
    ft_va.iloc[tr_va].reset_index(drop=True), y_va[tr_va], set(), top_k=10)
log(f'Top-10 VA features: {top_names_va}')

scaler_va = StandardScaler().fit(ft_va.iloc[tr_va][top_names_va].to_numpy(np.float32))
Xk_va = {s: scaler_va.transform(ft_va.iloc[ix][top_names_va].to_numpy(np.float32))
         for s, ix in [('train',tr_va),('dev',dv_va),('test',te_va)]}

tr_va_txt = df_va.iloc[tr_va][['text_clean']].reset_index(drop=True)
dv_va_txt = df_va.iloc[dv_va][['text_clean']].reset_index(drop=True)
te_va_txt = df_va.iloc[te_va][['text_clean']].reset_index(drop=True)
y_tr_va = y_va[tr_va]; y_dv_va = y_va[dv_va]; y_te_va = y_va[te_va]

feat_va = build_tfidf()
Xtr_va_t = feat_va.fit_transform(tr_va_txt); Xdv_va_t = feat_va.transform(dv_va_txt)
Xte_va_t = feat_va.transform(te_va_txt)
Xtr_va_h = hstack([Xtr_va_t, csr_matrix(Xk_va['train'])])
Xdv_va_h = hstack([Xdv_va_t, csr_matrix(Xk_va['dev'])])
Xte_va_h = hstack([Xte_va_t, csr_matrix(Xk_va['test'])])

C_GRID_VA = [0.0625, 0.125, 0.5, 1, 2, 4]
lr_va  = LogisticRegression(penalty='l2', solver='liblinear', class_weight='balanced', max_iter=2000)
svm_va = LinearSVC(class_weight='balanced', max_iter=5000)

for name, est, Xtr, Xdv, Xte in [
    ('VA_TFIDF_LR',   lr_va,  Xtr_va_t,Xdv_va_t,Xte_va_t),
    ('VA_TFIDF_SVC',  svm_va, Xtr_va_t,Xdv_va_t,Xte_va_t),
    ('VA_Hybrid_LR',  lr_va,  Xtr_va_h,Xdv_va_h,Xte_va_h),
    ('VA_Hybrid_SVC', svm_va, Xtr_va_h,Xdv_va_h,Xte_va_h),
]:
    run_linear_model(name,est,Xtr,y_tr_va,Xdv,y_dv_va,Xte,y_te_va,
                     C_GRID_VA,VA_REPORTS,VA_FIGS)

va_topk_pack = {'train':Xk_va['train'],'dev':Xk_va['dev'],'test':Xk_va['test']}
for base_name, hub in [
    ('VA_AraELECTRA','aubmindlab/araelectra-base-discriminator'),
    ('VA_MARBERT','UBC-NLP/MARBERT'),
]:
    for tag, pack in [('text',None),('top10',va_topk_pack)]:
        run_transformer(
            f'{base_name}_{tag}', hub,
            df_va.iloc[tr_va]['text_clean'].tolist(),
            df_va.iloc[dv_va]['text_clean'].tolist(),
            df_va.iloc[te_va]['text_clean'].tolist(),
            y_tr_va,y_dv_va,y_te_va, topk_pack=pack,
            report_dir=VA_REPORTS, fig_dir=VA_FIGS,
            ckpt_dir=os.path.join(VA_REPORTS,'checkpoints',f'{base_name}_{tag}'),
            max_len=192, epochs=4, lr=2e-5, bsz=32
        )
print('VERA-ARAB complete.')

---
AFND Dataset


606,912 articles; formal Modern Standard Arabic news. Key discriminative


AFND: Preprocess + Extract Features

In [ ]:
# ── Edit paths ───────────────────────────────────────────────────────────
AFND_FAKE_CSV  = '/content/AFND_fake.csv'
AFND_REAL_CSV  = '/content/AFND_real.csv'
AFND_TEXT_COL  = 'title'   # change to 'text' if your CSV uses that column
AFND_REPORTS   = '/content/reports/afnd'
AFND_FIGS      = '/content/figs/afnd'
makedirs(AFND_REPORTS, AFND_FIGS)
set_seed(42)

def load_afnd(path, label):
    df = read_csv_robust(path)
    df.columns = [c.strip() for c in df.columns]
    col = AFND_TEXT_COL if AFND_TEXT_COL in df.columns else df.columns[0]
    return pd.DataFrame({'text_raw': df[col], 'label': label})

df_afnd = pd.concat([load_afnd(AFND_FAKE_CSV,'Fake'),
                      load_afnd(AFND_REAL_CSV,'Real')], ignore_index=True)
df_afnd['text_clean'] = df_afnd['text_raw'].fillna('').apply(preprocess_arabic)
df_afnd = df_afnd[df_afnd['text_clean'].str.split().str.len()>2]
df_afnd = df_afnd.drop_duplicates(subset=['text_clean']).reset_index(drop=True)
df_afnd['y'] = df_afnd['label'].map({'Fake':1,'Real':0})

ft_afnd = pd.DataFrame([extract_features(t) for t in df_afnd['text_clean']])
log(f'AFND: {len(df_afnd):,} rows | {df_afnd["label"].value_counts().to_dict()}')

AFND: Train Linear + Transformer Models

In [ ]:
y_afnd = df_afnd['y'].to_numpy()
idx_afnd = np.arange(len(df_afnd))
tr_afnd, tmp_afnd = train_test_split(idx_afnd, test_size=0.20, stratify=y_afnd, random_state=42)
dv_afnd, te_afnd  = train_test_split(tmp_afnd, test_size=0.50, stratify=y_afnd[tmp_afnd], random_state=42)

top_names_afnd, _, _ = rank_features_by_mi(
    ft_afnd.iloc[tr_afnd].reset_index(drop=True), y_afnd[tr_afnd], set(), top_k=10)
log(f'Top-10 AFND features: {top_names_afnd}')

scaler_afnd = StandardScaler().fit(ft_afnd.iloc[tr_afnd][top_names_afnd].to_numpy(np.float32))
Xk_afnd = {s: scaler_afnd.transform(ft_afnd.iloc[ix][top_names_afnd].to_numpy(np.float32))
           for s,ix in [('train',tr_afnd),('dev',dv_afnd),('test',te_afnd)]}

tr_afnd_txt = df_afnd.iloc[tr_afnd][['text_clean']].reset_index(drop=True)
dv_afnd_txt = df_afnd.iloc[dv_afnd][['text_clean']].reset_index(drop=True)
te_afnd_txt = df_afnd.iloc[te_afnd][['text_clean']].reset_index(drop=True)
y_tr_afnd=y_afnd[tr_afnd]; y_dv_afnd=y_afnd[dv_afnd]; y_te_afnd=y_afnd[te_afnd]

feat_afnd = build_tfidf()
Xtr_afnd_t = feat_afnd.fit_transform(tr_afnd_txt)
Xdv_afnd_t = feat_afnd.transform(dv_afnd_txt)
Xte_afnd_t = feat_afnd.transform(te_afnd_txt)
Xtr_afnd_h = hstack([Xtr_afnd_t, csr_matrix(Xk_afnd['train'])])
Xdv_afnd_h = hstack([Xdv_afnd_t, csr_matrix(Xk_afnd['dev'])])
Xte_afnd_h = hstack([Xte_afnd_t, csr_matrix(Xk_afnd['test'])])

C_GRID_AFND=[0.0625,0.125,0.5,1,2]
lr_afnd  = LogisticRegression(penalty='l2',solver='liblinear',class_weight='balanced',max_iter=2000)
svm_afnd = LinearSVC(class_weight='balanced',max_iter=5000)

for name,est,Xtr,Xdv,Xte in [
    ('AFND_TFIDF_LR',  lr_afnd, Xtr_afnd_t,Xdv_afnd_t,Xte_afnd_t),
    ('AFND_TFIDF_SVC', svm_afnd,Xtr_afnd_t,Xdv_afnd_t,Xte_afnd_t),
    ('AFND_Hybrid_LR', lr_afnd, Xtr_afnd_h,Xdv_afnd_h,Xte_afnd_h),
    ('AFND_Hybrid_SVC',svm_afnd,Xtr_afnd_h,Xdv_afnd_h,Xte_afnd_h),
]:
    run_linear_model(name,est,Xtr,y_tr_afnd,Xdv,y_dv_afnd,Xte,y_te_afnd,
                     C_GRID_AFND,AFND_REPORTS,AFND_FIGS)

afnd_topk_pack={'train':Xk_afnd['train'],'dev':Xk_afnd['dev'],'test':Xk_afnd['test']}
for base_name,hub in [
    ('AFND_AraELECTRA','aubmindlab/araelectra-base-discriminator'),
    ('AFND_MARBERT','UBC-NLP/MARBERT'),
]:
    for tag,pack in [('text',None),('top10',afnd_topk_pack)]:
        run_transformer(
            f'{base_name}_{tag}',hub,
            df_afnd.iloc[tr_afnd]['text_clean'].tolist(),
            df_afnd.iloc[dv_afnd]['text_clean'].tolist(),
            df_afnd.iloc[te_afnd]['text_clean'].tolist(),
            y_tr_afnd,y_dv_afnd,y_te_afnd,topk_pack=pack,
            report_dir=AFND_REPORTS,fig_dir=AFND_FIGS,
            ckpt_dir=os.path.join(AFND_REPORTS,'checkpoints',f'{base_name}_{tag}'),
            max_len=192,epochs=4,lr=2e-5,bsz=32
        )
print('AFND complete.')

---
Unified Corpus Ablation Study

**Sections 3.5, 4.5, 4.6 of paper**  
Merges ANS + AFND + LIAR(AR) + WELFake(AR) + VERA-ARAB.  
Evaluates all model families; transformer results averaged over 3 seeds.  
Reports McNemar's exact test


Unified: Configuration & Output Directories

In [ ]:
# ── Edit path to the pre-merged CSV ──────────────────────────────────────
UNIFIED_RAW   = '/content/ANS_AFND_LIAR_WELFAKE_VERAARAB.csv'
UNIFIED_CLEAN = '/content/mega_clean.csv'
UNIFIED_FEAT  = '/content/mega_features.csv'

U_OUT     = '/content/reports_unified_joint'
U_CACHE   = os.path.join(U_OUT,'cache')
U_TABLES  = os.path.join(U_OUT,'tables')
U_FIGS    = os.path.join(U_OUT,'figures')
U_PREDS   = os.path.join(U_OUT,'predictions')
U_REPORTS = os.path.join(U_OUT,'reports')
U_CKPTS   = os.path.join(U_OUT,'checkpoints')
makedirs(U_OUT,U_CACHE,U_TABLES,U_FIGS,U_PREDS,U_REPORTS,U_CKPTS)

U_SEED   = 42
U_SEEDS  = [42, 52, 62]   # 3 seeds for transformer mean ± std
U_TOP_K  = 10
U_MAXLEN = 192
U_EPOCHS = 4
U_LR     = 2e-5
U_BSZ    = 64
U_GACC   = 2             # effective batch = 128

U_MODELS = [
    ('AraELECTRA', 'aubmindlab/araelectra-base-discriminator'),
    ('MARBERT',    'UBC-NLP/MARBERT'),
]
set_seed(U_SEED)
print('Unified corpus config set.')

 Unified: Load & Preprocess Corpus

In [ ]:
TEXT_CANDS  = ['text','headline','claim','content','article','title','tweet']
LABEL_CANDS = ['label','class','target','fake_flag','label_bin','veracity','truth']

def _find_col(df, candidates):
    for c in candidates:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in candidates): return c
    return None

if os.path.exists(UNIFIED_CLEAN) and os.path.exists(UNIFIED_FEAT):
    log('[SKIP] Loading cached unified corpus …')
    u_clean = pd.read_csv(UNIFIED_CLEAN)
    u_feats = pd.read_csv(UNIFIED_FEAT)
else:
    log('Building unified corpus from scratch …')
    raw = read_csv_robust(UNIFIED_RAW)
    raw.columns = [c.strip() for c in raw.columns]
    tc = _find_col(raw, TEXT_CANDS)
    lc = _find_col(raw, LABEL_CANDS)
    if tc is None or lc is None:
        raise RuntimeError(f'Cannot find text/label columns. Available: {list(raw.columns)}')
    df_u = raw[[tc,lc]].copy(); df_u.columns=['text','label_raw']
    df_u['label_str'] = df_u['label_raw'].apply(normalize_label)
    df_u = df_u.dropna(subset=['label_str']).reset_index(drop=True)
    df_u['clean_text'] = df_u['text'].apply(preprocess_arabic)
    df_u = (df_u[df_u['clean_text'].str.strip()!='']
              .pipe(lambda d: d[d['clean_text'].str.split().str.len()>2])
              .drop_duplicates(subset=['clean_text','label_str'])
              .reset_index(drop=True))
    df_u['label_bin'] = df_u['label_str'].map({'Fake':1,'Real':0}).astype(int)
    u_clean = df_u[['clean_text','label_str']].copy()
    feat_rows = [extract_features(t) for t in df_u['clean_text']]
    u_feats = pd.DataFrame(feat_rows)
    u_feats['label_str'] = df_u['label_str'].values
    u_feats['label_bin'] = df_u['label_bin'].values
    u_clean.to_csv(UNIFIED_CLEAN, index=False, encoding='utf-8-sig')
    u_feats.to_csv(UNIFIED_FEAT,  index=False, encoding='utf-8-sig')

log(f'Unified: {len(u_clean):,} rows | {u_clean["label_str"].value_counts().to_dict()}')

 Unified: Train/Dev/Test Split + Top-K Feature Ranking

In [ ]:
u_labels = u_clean['label_str'].map({'Fake':1,'Real':0}).astype(int).values
u_idx = np.arange(len(u_clean))
u_tr, u_tmp = train_test_split(u_idx, test_size=0.20, stratify=u_labels, random_state=U_SEED)
u_dv, u_te  = train_test_split(u_tmp, test_size=0.50, stratify=u_labels[u_tmp], random_state=U_SEED)

u_tr_txt = u_clean.iloc[u_tr]['clean_text'].tolist()
u_dv_txt = u_clean.iloc[u_dv]['clean_text'].tolist()
u_te_txt = u_clean.iloc[u_te]['clean_text'].tolist()
u_ytr = u_labels[u_tr].tolist()
u_ydv = u_labels[u_dv].tolist()
u_yte = u_labels[u_te].tolist()
log(f'Split — Train:{len(u_ytr):,}  Dev:{len(u_ydv):,}  Test:{len(u_yte):,}')

# Feature ranking on TRAIN only
U_EXCLUDE = {'label_str','label_bin'}
U_NUM_COLS = [c for c in u_feats.columns
              if c not in U_EXCLUDE and pd.api.types.is_numeric_dtype(u_feats[c])]
u_ftr_tr = u_feats.iloc[u_tr][U_NUM_COLS].fillna(0).reset_index(drop=True).astype(np.float32)
u_ftr_dv = u_feats.iloc[u_dv][U_NUM_COLS].fillna(0).reset_index(drop=True).astype(np.float32)
u_ftr_te = u_feats.iloc[u_te][U_NUM_COLS].fillna(0).reset_index(drop=True).astype(np.float32)

u_mi     = mutual_info_classif(u_ftr_tr.to_numpy(), np.array(u_ytr), random_state=U_SEED)
u_ranked = sorted(zip(U_NUM_COLS,u_mi), key=lambda x:x[1], reverse=True)
U_TOP10  = [n for n,_ in u_ranked[:U_TOP_K]]
U_MI_VALS= [v for _,v in u_ranked[:U_TOP_K]]
log(f'Top-{U_TOP_K} features (Unified): {U_TOP10}')

pd.DataFrame({'Rank':range(1,U_TOP_K+1),'Feature':U_TOP10,'MI':U_MI_VALS}).to_csv(
    os.path.join(U_TABLES,'top10_features.csv'), index=False)

u_scaler = StandardScaler().fit(u_ftr_tr[U_TOP10].to_numpy())
u_XtrK   = u_scaler.transform(u_ftr_tr[U_TOP10].to_numpy()).astype(np.float32)
u_XdvK   = u_scaler.transform(u_ftr_dv[U_TOP10].to_numpy()).astype(np.float32)
u_XteK   = u_scaler.transform(u_ftr_te[U_TOP10].to_numpy()).astype(np.float32)

# Plot Top-10 MI bar chart
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.barh(U_TOP10[::-1], U_MI_VALS[::-1], color='#2563EB', edgecolor='white')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top-10 Engineered Features (Unified Corpus)', fontweight='bold')
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(U_FIGS,'top10_mi.png'), dpi=180); plt.show()

Unified: TF-IDF Linear Ablation

In [ ]:
log('Building TF-IDF matrices …')
u_tfidf_w = TfidfVectorizer(analyzer='word',    ngram_range=(1,2), min_df=2, max_df=0.95)
u_tfidf_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,6), min_df=3, max_df=0.95)
u_Xtr_w = u_tfidf_w.fit_transform(u_tr_txt); u_Xtr_c = u_tfidf_c.fit_transform(u_tr_txt)
u_Xdv_w = u_tfidf_w.transform(u_dv_txt);    u_Xdv_c = u_tfidf_c.transform(u_dv_txt)
u_Xte_w = u_tfidf_w.transform(u_te_txt);    u_Xte_c = u_tfidf_c.transform(u_te_txt)
u_Xtr_tf = hstack([u_Xtr_w,u_Xtr_c]).tocsr()
u_Xdv_tf = hstack([u_Xdv_w,u_Xdv_c]).tocsr()
u_Xte_tf = hstack([u_Xte_w,u_Xte_c]).tocsr()
u_Xtr_fu = hstack([u_Xtr_tf,csr_matrix(u_XtrK)]).tocsr()
u_Xdv_fu = hstack([u_Xdv_tf,csr_matrix(u_XdvK)]).tocsr()
u_Xte_fu = hstack([u_Xte_tf,csr_matrix(u_XteK)]).tocsr()

u_lr_clf  = LogisticRegression(penalty='l2',solver='liblinear',C=1.0,
                                max_iter=3000,class_weight='balanced')
u_svm_clf = LinearSVC(C=1.0,class_weight='balanced',max_iter=5000)

u_linear_results = []

def run_linear_unified(name, clf, Xtr, Xte):
    pred_path = os.path.join(U_PREDS,f'{name}_predictions.csv')
    if os.path.exists(pred_path):
        log(f'  [CACHED] {name}')
        y_pred = pd.read_csv(pred_path)['y_pred'].tolist()
    else:
        clf.fit(Xtr, u_ytr)
        y_pred = clf.predict(Xte)
        pd.DataFrame({'y_true':u_yte,'y_pred':y_pred}).to_csv(pred_path,index=False)
    m  = compute_metrics(u_yte,y_pred)
    cm = confusion_matrix(u_yte,y_pred)
    rep= classification_report(u_yte,y_pred,digits=4,target_names=['Real','Fake'])
    _,ci_lo,ci_hi = bootstrap_ci(u_yte,y_pred)
    with open(os.path.join(U_REPORTS,f'{name}.txt'),'w') as f:
        f.write(f'{name}\n\n{rep}\nConfusion Matrix:\n{cm}\n')
    plot_confusion_matrix(cm,name,os.path.join(U_FIGS,f'{name}_cm.png'))
    log(f'  {name} | Acc={m["Accuracy"]:.4f} F1={m["F1 (macro)"]:.4f}')
    return {**m,'Model':name,'y_true':np.array(u_yte),'y_pred':np.array(y_pred),
            'CM':cm,'F1_CI_lo':ci_lo,'F1_CI_hi':ci_hi}

for name,clf,Xtr,Xte in [
    ('TFIDF_LogReg',          clone(u_lr_clf),  u_Xtr_tf,u_Xte_tf),
    ('TFIDF_LinearSVC',       clone(u_svm_clf), u_Xtr_tf,u_Xte_tf),
    ('FusionTop10_LogReg',    clone(u_lr_clf),  u_Xtr_fu,u_Xte_fu),
    ('FusionTop10_LinearSVC', clone(u_svm_clf), u_Xtr_fu,u_Xte_fu),
]:
    u_linear_results.append(run_linear_unified(name,clf,Xtr,Xte))

u_lin_df = pd.DataFrame([{k:v for k,v in r.items()
                           if k not in ('y_true','y_pred','CM')} for r in u_linear_results])
u_lin_df.to_csv(os.path.join(U_TABLES,'linear_summary.csv'),index=False)
print(u_lin_df[['Model','Accuracy','Precision (macro)','Recall (macro)','F1 (macro)']].to_string(index=False))

 Unified: Transformer Ablation (3 Seeds)

In [ ]:
# Runs AraELECTRA and MARBERT × {text-only, top10-fusion} × 3 seeds = 12 runs
# Each run is cached; restart and re-run to skip training and reload results.

u_topk_pack = {'train':u_XtrK,'dev':u_XdvK,'test':u_XteK}
u_transformer_results = []

for base_name, hub in U_MODELS:
    for s in U_SEEDS:
        for tag, pack in [('text',None),('top10',u_topk_pack)]:
            run_name = f'{base_name}_{tag}_seed{s}'
            ckpt_dir = os.path.join(U_CKPTS, run_name)
            pred_path= os.path.join(U_PREDS, f'{run_name}_predictions.csv')
            makedirs(ckpt_dir)
            set_seed(s)
            if os.path.exists(pred_path):
                log(f'  [CACHED] {run_name}')
                y_pred  = pd.read_csv(pred_path)['y_pred'].tolist()
                history = json.load(open(os.path.join(U_CACHE,f'{run_name}_history.json')))
                         if os.path.exists(os.path.join(U_CACHE,f'{run_name}_history.json')) else []
            else:
                log(f'  Training {run_name} …')
                tok = AutoTokenizer.from_pretrained(hub)
                def tok_fn(txts):
                    return tok(list(txts),padding=True,truncation=True,
                               max_length=U_MAXLEN,return_attention_mask=True)
                tr_enc=tok_fn(u_tr_txt); dv_enc=tok_fn(u_dv_txt); te_enc=tok_fn(u_te_txt)
                cc=pd.Series(u_ytr).value_counts().sort_index()
                cw=torch.tensor((cc.sum()/(2.0*cc)).to_numpy(dtype=np.float32))
                warmup=get_warmup_steps(len(u_ytr),U_BSZ,U_GACC,U_EPOCHS)
                if pack is None:
                    tr_ds=HFTextDataset(tr_enc,u_ytr); dv_ds=HFTextDataset(dv_enc,u_ydv)
                    te_ds=HFTextDataset(te_enc,u_yte)
                    cfg_m=AutoConfig.from_pretrained(hub,num_labels=2,
                                                      id2label={0:'real',1:'fake'},
                                                      label2id={'real':0,'fake':1})
                    model=AutoModelForSequenceClassification.from_pretrained(hub,config=cfg_m)
                else:
                    tr_ds=HFTextDataset(tr_enc,u_ytr,feats=pack['train'])
                    dv_ds=HFTextDataset(dv_enc,u_ydv,feats=pack['dev'])
                    te_ds=HFTextDataset(te_enc,u_yte,feats=pack['test'])
                    model=FusionClassifier(hub,pack['train'].shape[1],num_labels=2,class_weights=cw)
                hist_cb=HistoryCallback()
                ta=TrainingArguments(
                    output_dir=ckpt_dir,num_train_epochs=U_EPOCHS,
                    per_device_train_batch_size=U_BSZ,per_device_eval_batch_size=U_BSZ,
                    gradient_accumulation_steps=U_GACC,learning_rate=U_LR,
                    weight_decay=0.01,warmup_steps=warmup,
                    save_strategy='epoch',**{EVAL_KEY:'epoch'},
                    load_best_model_at_end=True,metric_for_best_model='eval_loss',
                    greater_is_better=False,save_total_limit=2,logging_steps=200,
                    report_to=[],fp16=torch.cuda.is_available() and (pack is None),
                )
                trainer=Trainer(model=model,args=ta,train_dataset=tr_ds,
                                eval_dataset=dv_ds,callbacks=[hist_cb])
                trainer.train()
                pred=trainer.predict(te_ds)
                y_pred=pred.predictions.argmax(axis=-1).tolist()
                pd.DataFrame({'y_true':u_yte,'y_pred':y_pred}).to_csv(pred_path,index=False)
                save_json(os.path.join(U_CACHE,f'{run_name}_history.json'),hist_cb.history)
                history=hist_cb.history
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()

            m  = compute_metrics(u_yte,y_pred)
            cm = confusion_matrix(u_yte,y_pred)
            rep= classification_report(u_yte,y_pred,digits=4,target_names=['Real','Fake'])
            _,ci_lo,ci_hi = bootstrap_ci(u_yte,y_pred)
            with open(os.path.join(U_REPORTS,f'{run_name}.txt'),'w') as f:
                f.write(f'{run_name}\n\n{rep}\nConfusion Matrix:\n{cm}\n')
            plot_confusion_matrix(cm,run_name,os.path.join(U_FIGS,f'{run_name}_cm.png'))
            log(f'    Acc={m["Accuracy"]:.4f}  F1={m["F1 (macro)"]:.4f}')
            u_transformer_results.append({
                'RunName':run_name,'Backbone':base_name,'Setting':tag,'Seed':s,
                **m,'y_true':np.array(u_yte),'y_pred':np.array(y_pred),
                'CM':cm,'F1_CI_lo':ci_lo,'F1_CI_hi':ci_hi,'history':history,
            })

print(f'All {len(u_transformer_results)} transformer runs complete.')

 Unified: McNemar Tests + Summary Tables

In [ ]:
# McNemar's exact test for every pairwise comparison (Table 16 in paper)

u_yt_ = np.array(u_yte)
u_pred_map = {r['Model']:r for r in u_linear_results}
u_pred_map.update({r['RunName']:r for r in u_transformer_results})

def u_mv_by(backbone, setting):
    return majority_vote([r['y_pred'] for r in u_transformer_results
                          if r['Backbone']==backbone and r['Setting']==setting])

mcn_rows = []
for a,b,la,lb in [
    ('TFIDF_LogReg','FusionTop10_LogReg','TF-IDF+LR','Hybrid+LR'),
    ('TFIDF_LinearSVC','FusionTop10_LinearSVC','TF-IDF+SVC','Hybrid+SVC'),
    ('TFIDF_LogReg','TFIDF_LinearSVC','TF-IDF+LR','TF-IDF+SVC'),
    ('FusionTop10_LogReg','FusionTop10_LinearSVC','Hybrid+LR','Hybrid+SVC'),
]:
    res=mcnemar_exact(u_pred_map[a]['y_true'],u_pred_map[a]['y_pred'],u_pred_map[b]['y_pred'])
    dF1=round(u_pred_map[b]['F1 (macro)']-u_pred_map[a]['F1 (macro)'],4)
    mcn_rows.append({'Category':'Linear','Comparison':f'{la} vs {lb}','ΔF1':dF1,**res})

for bb in ['AraELECTRA','MARBERT']:
    mv_t=u_mv_by(bb,'text'); mv_k=u_mv_by(bb,'top10')
    res=mcnemar_exact(u_yt_,mv_t,mv_k)
    dF1=round(f1_score(u_yt_,mv_k,average='macro',zero_division=0)-
              f1_score(u_yt_,mv_t,average='macro',zero_division=0),4)
    mcn_rows.append({'Category':'Transformer','Comparison':f'{bb}: text-MV vs top10-MV','ΔF1':dF1,**res})

mv_e=u_mv_by('AraELECTRA','top10'); mv_m=u_mv_by('MARBERT','top10')
res=mcnemar_exact(u_yt_,mv_e,mv_m)
dF1=round(f1_score(u_yt_,mv_m,average='macro',zero_division=0)-
          f1_score(u_yt_,mv_e,average='macro',zero_division=0),4)
mcn_rows.append({'Category':'Cross-model','Comparison':'AraELECTRA-top10 vs MARBERT-top10','ΔF1':dF1,**res})

mcn_df = pd.DataFrame(mcn_rows)[['Category','Comparison','ΔF1','b','c','n_discordant','p_value','significant']]
mcn_df.to_csv(os.path.join(U_TABLES,'mcnemar_all.csv'),index=False)

# Transformer mean ± std summary
trans_rows = [{'Backbone':r['Backbone'],'Setting':r['Setting'],'Seed':r['Seed'],
               'Accuracy':r['Accuracy'],'F1 (macro)':r['F1 (macro)']}
              for r in u_transformer_results]
trans_df = pd.DataFrame(trans_rows)
u_trans_summary = (trans_df.groupby(['Backbone','Setting'])
                   .agg(Acc_mean=('Accuracy','mean'),Acc_std=('Accuracy','std'),
                         F1_mean=('F1 (macro)','mean'),F1_std=('F1 (macro)','std'))
                   .reset_index())
u_trans_summary.to_csv(os.path.join(U_TABLES,'trans_mean_std.csv'),index=False)

print('\n── Linear Model Summary ──')
print(u_lin_df[['Model','Accuracy','F1 (macro)']].to_string(index=False))
print('\n── Transformer Mean ± Std ──')
for _,row in u_trans_summary.iterrows():
    print(f"{row['Backbone']:12s} ({row['Setting']:5s}) | Acc={row['Acc_mean']:.4f}±{row['Acc_std']:.4f}  F1={row['F1_mean']:.4f}±{row['F1_std']:.4f}")
print('\n── McNemar Tests ──')
print(mcn_df.to_string(index=False))

Unified: Final 4-Panel Overview Figure

In [ ]:
PALETTE = ['#2563EB','#16A34A','#DC2626','#D97706','#7C3AED','#0891B2','#BE185D','#059669']

fig = plt.figure(figsize=(16,10))
gs  = gridspec.GridSpec(2,3,figure=fig,hspace=0.50,wspace=0.38)

# (A) Linear F1
ax_a = fig.add_subplot(gs[0,0])
LABEL_MAP={'TFIDF_LogReg':'TF-IDF+LR','TFIDF_LinearSVC':'TF-IDF+SVC',
           'FusionTop10_LogReg':'Hybrid+LR','FusionTop10_LinearSVC':'Hybrid+SVC'}
labs_a=[LABEL_MAP.get(r['Model'],r['Model']) for r in u_linear_results]
f1s_a =[r['F1 (macro)'] for r in u_linear_results]
ax_a.barh(labs_a[::-1],f1s_a[::-1],color=PALETTE[:4][::-1],edgecolor='white')
ax_a.set_xlabel('Macro-F1'); ax_a.set_title('(A) Linear Ablation',fontweight='bold')

# (B) Transformer F1 mean ± std
ax_b = fig.add_subplot(gs[0,1:])
x_b  = np.arange(len(u_trans_summary))
c_b  = [PALETTE[0] if s=='text' else PALETTE[1] for s in u_trans_summary['Setting']]
ax_b.bar(x_b,u_trans_summary['F1_mean'],yerr=u_trans_summary['F1_std'],
         capsize=6,color=c_b,edgecolor='white')
ax_b.set_xticks(x_b)
ax_b.set_xticklabels([f"{r.Backbone}\n({r.Setting})" for r in u_trans_summary.itertuples()],fontsize=9)
ax_b.set_ylabel('Macro-F1 (mean ± std)')
ax_b.set_title('(B) Transformer Ablation (3 seeds)',fontweight='bold')
ax_b.legend(handles=[Patch(color=PALETTE[0],label='Text only'),
                      Patch(color=PALETTE[1],label='Top-10 fusion')])

# (C) McNemar -log10(p)
ax_c = fig.add_subplot(gs[1,:2])
pvals=mcn_df['p_value'].tolist()
ax_c.barh(mcn_df['Comparison'].tolist()[::-1],
          [-np.log10(max(p,1e-300)) for p in pvals[::-1]],
          color=[PALETTE[2] if p<0.05 else PALETTE[7] for p in pvals[::-1]],
          edgecolor='white')
ax_c.axvline(-np.log10(0.05),color='red',ls='--',lw=1.5,label='p=0.05')
ax_c.set_xlabel('−log₁₀(p)'); ax_c.set_title('(C) McNemar Significance',fontweight='bold')
ax_c.legend(handles=[Patch(color=PALETTE[2],label='p<0.05 Significant'),
                      Patch(color=PALETTE[7],label='Not significant')])

# (D) Top-10 MI scores
ax_d = fig.add_subplot(gs[1,2])
ax_d.barh(U_TOP10[::-1],U_MI_VALS[::-1],color=PALETTE[4],edgecolor='white')
ax_d.set_xlabel('MI Score'); ax_d.set_title('(D) Top-10 Features',fontweight='bold')

plt.suptitle('Unified Dataset — Ablation Study Overview',fontsize=14,fontweight='bold',y=1.01)
fig_path = os.path.join(U_FIGS,'overview_4panel.png')
plt.savefig(fig_path,dpi=180,bbox_inches='tight')
plt.show()
print(f'Figure saved to: {fig_path}')

---
## ✅ Pipeline Complete

All outputs are saved under `/content/`:

